# TASK 1 - Swimming


To classify the two new examples, the Naïve Bayes decision rule was applied. First, I computed the prior probabilities of the target classes (Swimming = Yes/No) from the dataset. Next, the conditional probabilities were calculated for each feature.

Because some feature values do not appear in one of the classes, the basic Naïve Bayes model produces zero probabilities, which would force the entire likelihood to zero. To avoid this, we applied Laplace (add-1) smoothing, ensuring that no conditional probability equals zero.

Finally, for each example the posterior scores were computed and assigned the class with the higher posterior score (MAP decision rule).

Based on these calculations,

✅ the answer is:


**X1 is classified as Swimming = Yes**

**X2 is classified as Swimming = No**

In [1]:
import pandas as pd

data = [
    ["Light","Light","Cold","Light","None","No"],
    ["Light","Moderate","Cold","Moderate","None","No"],
    ["Moderate","Light","Cold","Light","None","No"],
    ["Heavy","Light","Warm","Light","Some","Yes"],
    ["Heavy","Moderate","Warm","Moderate","Some","Yes"],
    ["Moderate","Moderate","Cold","Moderate","None","No"],
    ["Moderate","Heavy","Warm","Moderate","Some","Yes"],
    ["Light","Moderate","Cold","Gale","None","No"],
    ["Light","Light","Cold","Moderate","None","No"],
    ["Heavy","Moderate","Warm","Light","Some","Yes"],
]

cols = ["RR","RT","Temp","Wind","Sunshine","Swimming"]

df = pd.DataFrame(data, columns=cols)
df

,RR,RT,Temp,Wind,Sunshine,Swimming
0,Light,Light,Cold,Light,None,No
1,Light,Moderate,Cold,Moderate,None,No
2,Moderate,Light,Cold,Light,None,No
3,Heavy,Light,Warm,Light,Some,Yes
4,Heavy,Moderate,Warm,Moderate,Some,Yes
5,Moderate,Moderate,Cold,Moderate,None,No
6,Moderate,Heavy,Warm,Moderate,Some,Yes
7,Light,Moderate,Cold,Gale,None,No
8,Light,Light,Cold,Moderate,None,No
9,Heavy,Moderate,Warm,Light,Some,Yes


In [3]:
# Compute prior probabilities P(Yes) and P(No) from the dataset

df["Swimming"].value_counts(normalize=True)

,proportion
Swimming,
No,0.6
Yes,0.4


In [10]:
# Likelihood P(X | Yes)

likelihood_yes = (
    RR_yes["heavy"] *
    RT_yes["moderate"] *
    T_yes["warm"] *
    W_yes["light"] *
    S_yes["some"]
)

# Likelihood P(X | No)
likelihood_no = (
    RR_no["heavy"] *
    RT_no["moderate"] *
    T_no["warm"] *
    W_no["light"] *
    S_no["some"]
)

likelihood_yes, likelihood_no

(0.046875, 0.0)

In [13]:
# We apply Naïve Bayes classification using Laplace smoothing to avoid zero probabilities. For each class, we multiply the prior probability by the smoothed likelihoods of all observed feature values to obtain posterior scores, and the class with the higher posterior score is selected as the prediction.

import numpy as np

# ---------- 1) Priors ----------
prior_yes = 4/10
prior_no  = 6/10

# ---------- 2) Counts from the contingency table (numerator/denominator) ----------
# Denominators: Yes -> 4, No -> 6  (from the table)
N_yes = 4
N_no  = 6

# Feature value counts (numerators)
RR_yes = {"light":0, "moderate":2, "heavy":2}
RR_no  = {"light":3, "moderate":3, "heavy":0}

RT_yes = {"light":1, "moderate":2, "heavy":1}
RT_no  = {"light":3, "moderate":3, "heavy":0}

T_yes  = {"cold":1, "warm":3}
T_no   = {"cold":5, "warm":1}

W_yes  = {"light":2, "moderate":2, "gale":0}
W_no   = {"light":2, "moderate":2, "gale":2}

S_yes  = {"some":2, "none":2}
S_no   = {"some":4, "none":2}

# How many possible values per feature (k)
k_RR = 3
k_RT = 3
k_T  = 2
k_W  = 3
k_S  = 2

# ---------- 3) Laplace smoothing ----------
def laplace_prob(count, total, k):
    # P = (count + 1) / (total + k)
    return (count + 1) / (total + k)

def score_example(RR, RT, T, W, S):
    # Likelihoods with Laplace smoothing (no more zeros)
    lik_yes = (
        laplace_prob(RR_yes[RR], N_yes, k_RR) *
        laplace_prob(RT_yes[RT], N_yes, k_RT) *
        laplace_prob(T_yes[T],   N_yes, k_T)  *
        laplace_prob(W_yes[W],   N_yes, k_W)  *
        laplace_prob(S_yes[S],   N_yes, k_S)
    )

    lik_no = (
        laplace_prob(RR_no[RR], N_no, k_RR) *
        laplace_prob(RT_no[RT], N_no, k_RT) *
        laplace_prob(T_no[T],   N_no, k_T)  *
        laplace_prob(W_no[W],   N_no, k_W)  *
        laplace_prob(S_no[S],   N_no, k_S)
    )

    # Posterior scores (proportional)
    post_yes = prior_yes * lik_yes
    post_no  = prior_no  * lik_no

    pred = "Yes" if post_yes > post_no else "No"

    return {
        "lik_yes": lik_yes, "lik_no": lik_no,
        "post_yes": post_yes, "post_no": post_no,
        "pred": pred
    }

# ---------- 4) Classify X1 and X2 ----------
# X1: RR=Heavy, RT=Moderate, T=Warm, W=Light, S=Some
res_X1 = score_example("heavy", "moderate", "warm", "light", "some")

# X2: RR=Light, RT=Moderate, T=Warm, W=Light, S=Some
res_X2 = score_example("light", "moderate", "warm", "light", "some")

print("X1 ->", res_X1)
print("X2 ->", res_X2)

X1 -> {'lik_yes': 0.02623906705539358, 'lik_no': 0.0025720164609053494, 'post_yes': 0.010495626822157433, 'post_no': 0.0015432098765432096, 'pred': 'Yes'}
X2 -> {'lik_yes': 0.008746355685131192, 'lik_no': 0.010288065843621397, 'post_yes': 0.003498542274052477, 'post_no': 0.006172839506172838, 'pred': 'No'}


In [15]:
#Compare posterior scores for each class and assign the class with the higher posterior probability as the final prediction (MAP decision rule).

prediction_X1 = "Yes" if res_X1["post_yes"] > res_X1["post_no"] else "No"
prediction_X2 = "Yes" if res_X2["post_yes"] > res_X2["post_no"] else "No"

print("Prediction X1:", prediction_X1)
print("Prediction X2:", prediction_X2)

Prediction X1: Yes
Prediction X2: No


# TASK 2 - Sunburn

For the example X = (Hair=blonde, Height=average, Build=heavy, Lotion=no):
Naïve Bayes computes the posterior scores for each class:
	•	Score(sunburned) = 0.0556
	•	Score(none) = 0.0040

After normalisation the classifier is:

✅ **Prediction: sunburned (≈93% probability)**

In [16]:
import pandas as pd

sunburn = pd.DataFrame({
    "Hair":   ["blonde","blonde","brown","blonde","red","brown","brown","brown"],
    "Height": ["average","tall","short","short","average","tall","average","short"],
    "Build":  ["light","average","average","average","heavy","heavy","heavy","light"],
    "Lotion": ["no","yes","yes","no","no","no","no","yes"],
    "Result": ["sunburned","none","none","sunburned","sunburned","none","none","none"]
})

sunburn

,Hair,Height,Build,Lotion,Result
0,blonde,average,light,no,sunburned
1,blonde,tall,average,yes,none
2,brown,short,average,yes,none
3,blonde,short,average,no,sunburned
4,red,average,heavy,no,sunburned
5,brown,tall,heavy,no,none
6,brown,average,heavy,no,none
7,brown,short,light,yes,none


In [17]:
# counts of classes
class_counts = sunburn["Result"].value_counts()

# prior probabilities
prior = class_counts / len(sunburn)

class_counts, prior

(Result
 none         5
 sunburned    3
 Name: count, dtype: int64,
 Result
 none         0.625
 sunburned    0.375
 Name: count, dtype: float64)

In [21]:
# Generate contingency tables for each feature against the target class to obtain frequency counts needed for conditional probability calculations in Naïve Bayes.

features = ["Hair", "Height", "Build", "Lotion"]

for f in features:
    print(f"\n--- {f} ---")
    print(pd.crosstab(sunburn[f], sunburn["Result"]))


--- Hair ---
Result  none  sunburned
Hair                   
blonde     1          2
brown      4          0
red        0          1

--- Height ---
Result   none  sunburned
Height                  
average     1          2
short       2          1
tall        2          0

--- Build ---
Result   none  sunburned
Build                   
average     2          1
heavy       2          1
light       1          1

--- Lotion ---
Result  none  sunburned
Lotion                 
no         2          3
yes        3          0


In [26]:
# Normalises the previously created contingency tables to convert frequency counts into conditional probabilities P(feature=value \mid class), and combines them into a single likelihood table used by the Naïve Bayes classifier.

# classes
classes = ["sunburned", "none"]
features = ["Hair", "Height", "Build", "Lotion"]

tables = []

for f in features:
    # counts
    ct = pd.crosstab(sunburn[f], sunburn["Result"])

    # convert to probabilities P(feature=value | class)
    probs = ct.div(ct.sum(axis=0), axis=1)

    # add feature name as first index level
    probs["Feature"] = f
    probs["Value"] = probs.index

    tables.append(probs.reset_index(drop=True))

# merge all into one table
likelihood_table = pd.concat(tables)
likelihood_table = likelihood_table[["Feature","Value","sunburned","none"]]

likelihood_table

Result,Feature,Value,sunburned,none
0,Hair,blonde,0.666667,0.2
1,Hair,brown,0.000000,0.8
2,Hair,red,0.333333,0.0
0,Height,average,0.666667,0.2
1,Height,short,0.333333,0.4
2,Height,tall,0.000000,0.4
0,Build,average,0.333333,0.4
1,Build,heavy,0.333333,0.4
2,Build,light,0.333333,0.2
0,Lotion,no,1.000000,0.4


In [32]:
# This code applies the Naïve Bayes classifier to the new example X. It multiplies the conditional probabilities P(feature=value \mid class) to obtain the likelihood for each class, combines them with the prior probabilities, then normalises the results to compute the posterior probabilities P(class \mid X). The class with the higher posterior probability is returned as the prediction.

# Example X
X = {"Hair":"blonde", "Height":"average", "Build":"heavy", "Lotion":"no"}

# --- likelihoods P(X|class) ---
lik_s, lik_n = 1.0, 1.0

for f, val in X.items():
    row = likelihood_table[(likelihood_table["Feature"]==f) & (likelihood_table["Value"]==val)]
    lik_s *= row["sunburned"].values[0]
    lik_n *= row["none"].values[0]

# --- priors ---
prior_s = 3/8
prior_n = 5/8

# --- unnormalised posteriors ---
post_s = lik_s * prior_s
post_n = lik_n * prior_n

# --- NORMALISATION (real probabilities) ---
total = post_s + post_n
prob_s = post_s / total
prob_n = post_n / total

prediction = "sunburned" if prob_s > prob_n else "none"

print("P(sunburned | X) =", round(prob_s,3))
print("P(none | X)      =", round(prob_n,3))
print("Prediction:", prediction)

P(sunburned | X) = 0.933
P(none | X)      = 0.067
Prediction: sunburned


# TASK 3 - Credit Risk

Using the Naïve Bayes classifier for the new application X = (Credit_History = bad, Debt = low, Income = 30to60):

The posterior probabilities are:
	•	P(high \mid X) \approx 0.492
	•	P(medium \mid X) \approx 0.443
	•	P(low \mid X) \approx 0.066

Since the highest posterior probability corresponds to high risk, the classifier predicts:

✅ **Predicted risk level: high**

In [34]:
import pandas as pd

data = [
    ["bad","low","0to30","high"],
    ["bad","high","30to60","high"],
    ["bad","low","0to30","high"],
    ["unknown","high","30to60","high"],
    ["unknown","high","0to30","high"],
    ["good","high","0to30","high"],
    ["bad","low","over60","medium"],
    ["unknown","low","30to60","medium"],
    ["good","high","30to60","medium"],
    ["unknown","low","over60","low"],
    ["unknown","low","over60","low"],
    ["good","low","over60","low"],
    ["good","high","over60","low"],
    ["good","high","over60","low"]
]

credit = pd.DataFrame(data, columns=["Credit_History","Debt","Income","Risk"])

credit

,Credit_History,Debt,Income,Risk
0,bad,low,0to30,high
1,bad,high,30to60,high
2,bad,low,0to30,high
3,unknown,high,30to60,high
4,unknown,high,0to30,high
5,good,high,0to30,high
6,bad,low,over60,medium
7,unknown,low,30to60,medium
8,good,high,30to60,medium
9,unknown,low,over60,low


In [35]:
# Probability caclulation - frequency (how ofte=n) of each class in the dataset: P(Risk = high),; P(Risk = medium),; P(Risk = low)

# class prior probabilities
priors = credit["Risk"].value_counts(normalize=True)

priors

,proportion
Risk,
high,0.428571
low,0.357143
medium,0.214286


In [38]:
# Generate contingency tables for each feature against the target class (Risk)
# These tables contain frequency counts needed to compute conditional probabilities P(feature | class)

features = ["Credit_History", "Debt", "Income"]

for f in features:
    print(f"\n--- {f} ---")
    print(pd.crosstab(credit[f], credit["Risk"]))


--- Credit_History ---
Risk            high  low  medium
Credit_History                   
bad                3    0       1
good               1    3       1
unknown            2    2       1

--- Debt ---
Risk  high  low  medium
Debt                   
high     4    2       1
low      2    3       2

--- Income ---
Risk    high  low  medium
Income                   
0to30      4    0       0
30to60     2    0       2
over60     0    5       1


In [45]:
# Creates contingency tables for each feature against the target class Risk and normalises column-wise to obtain conditional probabilities P(feature=value \mid Risk) required for the Naïve Bayes classifier.

# Conditional probabilities P(feature=value | class)
cond_probs = {}

features = ["Credit_History", "Debt", "Income"]

for f in features:
    table = pd.crosstab(credit[f], credit["Risk"])
    cond_probs[f] = table.div(table.sum(axis=0), axis=1)

# show all tables
for f in features:
    print(f"\n=== {f} ===")
    print(cond_probs[f])


=== Credit_History ===
Risk                high  low    medium
Credit_History                         
bad             0.500000  0.0  0.333333
good            0.166667  0.6  0.333333
unknown         0.333333  0.4  0.333333

=== Debt ===
Risk      high  low    medium
Debt                         
high  0.666667  0.4  0.333333
low   0.333333  0.6  0.666667

=== Income ===
Risk        high  low    medium
Income                         
0to30   0.666667  0.0  0.000000
30to60  0.333333  0.0  0.666667
over60  0.000000  1.0  0.333333


In [49]:
features = ["Credit_History", "Debt", "Income"]
likelihoods = {}

for f in features:
    table = pd.crosstab(credit[f], credit["Risk"])
    table = table + 1  # Laplace smoothing
    likelihoods[f] = table.div(table.sum(axis=0), axis=1)

In [50]:
X = {
    "Credit_History": "bad",
    "Debt": "low",
    "Income": "30to60"
}

classes = ["high", "medium", "low"]
post = {}

for c in classes:
    p = priors[c]

    for f, val in X.items():
        p *= likelihoods[f].loc[val, c]

    post[c] = p

post

{'high': np.float64(0.023809523809523808),
 'medium': np.float64(0.021428571428571425),
 'low': np.float64(0.0031887755102040817)}

In [53]:
# New example
X = {"Credit_History":"bad", "Debt":"low", "Income":"30to60"}

# Calculate posterior (unnormalised)
classes = priors.index.tolist()
post = {}

for c in classes:
    p = priors[c]                     # P(class)
    for f, val in X.items():
        p *= likelihoods[f].loc[val, c]   # P(feature=value | class)
    post[c] = p

# Prediction (argmax)
prediction = max(post, key=post.get)

print("Unnormalised posteriors:", post)
print("Prediction:", prediction)


# Normalise probabilities
total = sum(post.values())
post_norm = {c: post[c]/total for c in post}

print("Normalised posteriors:", {k: round(v,3) for k,v in post_norm.items()})

Unnormalised posteriors: {'high': np.float64(0.023809523809523808), 'low': np.float64(0.0031887755102040817), 'medium': np.float64(0.021428571428571425)}
Prediction: high
Normalised posteriors: {'high': np.float64(0.492), 'low': np.float64(0.066), 'medium': np.float64(0.442)}


# TASK 4 - Athletes

**4-A.** Given the nature of the AthleteSelection dataset, the most appropriate Naïve Bayes model in scikit-learn is ✅  **GaussianNB**.

This is because the predictive features (Speed and Agility) are continuous numerical variables rather than categorical counts.
Gaussian Naïve Bayes assumes that, for each class, feature values follow a normal (Gaussian) distribution and estimates the mean and variance of each feature per class. This matches the statistical structure of the dataset, whereas MultinomialNB and BernoulliNB are designed for discrete frequency or binary data.

In [54]:
from google.colab import files
import pandas as pd

# upload AthleteSelection.csv
files.upload()

# read dataset
athlete = pd.read_csv("AthleteSelection.csv")

athlete.head()

Saving AthleteSelection.csv to AthleteSelection.csv


,Athlete,Speed,Agility,Selected
0,x1,2.50,6.00,No
1,x2,3.75,8.00,No
2,x3,2.25,5.50,No
3,x4,3.25,8.25,No
4,x5,2.75,7.50,No


In [55]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB

# read training data
athlete = pd.read_csv("AthleteSelection.csv")

# X = features, y = target
X_train = athlete[["Speed", "Agility"]].values
y_train = athlete["Selected"].values   # "Yes"/"No"

# train model
nb = GaussianNB()
nb.fit(X_train, y_train)

# quick sanity check
print("Classes:", nb.classes_)
print("Train size:", X_train.shape)

Classes: ['No' 'Yes']
Train size: (20, 2)


In [57]:
from google.colab import files
import pandas as pd

files.upload()          # обери AthleteTest.csv
test = pd.read_csv("AthleteTest.csv")

test.head()

Saving AthleteTest.csv to AthleteTest.csv


,Athlete,Speed,Agility
0,t1,3.3,8.2
1,t2,4.5,4.5
2,t3,5.5,7.2
3,t4,3.8,8.8
4,t5,5.5,5.2


In [58]:
# load test data
test = pd.read_csv("AthleteTest.csv")

# take same features as training
X_test = test[["Speed", "Agility"]].values

# apply classifier (predict class)
predictions = nb.predict(X_test)

# show results
test["Predicted"] = predictions
test

,Athlete,Speed,Agility,Predicted
0,t1,3.3,8.2,No
1,t2,4.5,4.5,No
2,t3,5.5,7.2,Yes
3,t4,3.8,8.8,No
4,t5,5.5,5.2,Yes
5,t6,8.1,7.8,Yes
6,t7,7.7,5.2,Yes
7,t8,6.1,5.5,Yes
8,t9,5.5,6.0,Yes
9,t10,6.1,5.5,Yes


In [59]:
# probability of each class
probs = nb.predict_proba(X_test)

# adding probability to be Selected = Yes
test["P(Selected=Yes)"] = probs[:, list(nb.classes_).index("Yes")]

test

,Athlete,Speed,Agility,Predicted,P(Selected=Yes)
0,t1,3.3,8.2,No,0.041314
1,t2,4.5,4.5,No,0.122983
2,t3,5.5,7.2,Yes,0.911933
3,t4,3.8,8.8,No,0.150478
4,t5,5.5,5.2,Yes,0.799833
5,t6,8.1,7.8,Yes,0.999997
6,t7,7.7,5.2,Yes,0.999945
7,t8,6.1,5.5,Yes,0.972931
8,t9,5.5,6.0,Yes,0.854283
9,t10,6.1,5.5,Yes,0.972931


In [60]:
# rank by probability of being selected
ranking = test.sort_values("P(Selected=Yes)", ascending=False)

ranking

,Athlete,Speed,Agility,Predicted,P(Selected=Yes)
5,t6,8.1,7.8,Yes,0.999997
6,t7,7.7,5.2,Yes,0.999945
7,t8,6.1,5.5,Yes,0.972931
9,t10,6.1,5.5,Yes,0.972931
2,t3,5.5,7.2,Yes,0.911933
8,t9,5.5,6.0,Yes,0.854283
4,t5,5.5,5.2,Yes,0.799833
3,t4,3.8,8.8,No,0.150478
1,t2,4.5,4.5,No,0.122983
0,t1,3.3,8.2,No,0.041314


**4-B.** Predict_proba returns the full set of class probabilities produced by the Naïve Bayes model for each athlete, i.e. the probability of both outcomes (Selected = Yes and Selected = No).

The column P(Selected = Yes) is not a new computation — it simply extracts the probability of the positive class (“Yes”) from those results and stores it as a single confidence score.
This value is then used to rank athletes by their likelihood of being selected.

✅  The athlete with **the highest probability** of being selected is **t3 (≈ 0.912).**

✅  The athlete with **the lowest probability** of being selected is **t1 (≈ 0.041).**

In [61]:
# Most likely to be selected
ranking.iloc[0]

,5
Athlete,t6
Speed,8.1
Agility,7.8
Predicted,Yes
P(Selected=Yes),0.999997


In [63]:
# Least likely to be selected
ranking.iloc[-1]

,0
Athlete,t1
Speed,3.3
Agility,8.2
Predicted,No
P(Selected=Yes),0.041314


**4С. **The GaussianNB model stores two parameters for each class and feature:

- theta_ — the mean value of the feature within the class
- var_ — the variance of the feature within the class

After computing the means and variances manually using pandas (with ddof = 0), ✅ **the obtained values exactly match the parameters learned by the model.**

Therefore, the Gaussian Naïve Bayes classifier estimates the parameters of the Gaussian distribution for each feature and class directly from the training data using maximum likelihood estimation.

This confirms that the model internally fits a normal distribution to each feature per class and uses these distributions to compute posterior probabilities.


In [64]:
# Step 1: inspect what GaussianNB learned
# theta_ = means per class per feature
# var_   = variances per class per feature

print("Classes order:", nb.classes_)      # which row is 'No' and which is 'Yes'
print("theta_ (means):\n", nb.theta_)
print("var_ (variances):\n", nb.var_)

Classes order: ['No' 'Yes']
theta_ (means):
 [[3.39583333 5.08333333]
 [6.40625    6.96875   ]]
var_ (variances):
 [[0.80685764 3.99305556]
 [1.37402344 3.91308594]]


In [65]:
# Step 2: manual estimates using pandas

# means
means = athlete.groupby("Selected")[["Speed","Agility"]].mean()

# variances (ddof=0 !!! важливо — як у GaussianNB)
vars_ = athlete.groupby("Selected")[["Speed","Agility"]].var(ddof=0)

print("Manual means:\n", means)
print("\nManual variances:\n", vars_)

Manual means:
              Speed   Agility
Selected                    
No        3.395833  5.083333
Yes       6.406250  6.968750

Manual variances:
              Speed   Agility
Selected                    
No        0.806858  3.993056
Yes       1.374023  3.913086


# TASK 5 - Antibody Tests

**5.** When the infection prevalence decreases from 5% to 1% while the test sensitivity and specificity remain unchanged, the posterior probability of infection given a positive test decreases substantially. Consequently, ✅  **the proportion of false positives among positive test results increases from 27.9% to 66.9%**, illustrating the base-rate effect in Bayesian inference

In [66]:
# 1) Put the contingency table into Colab (as counts)
import pandas as pd

N = 10_000

# counts from the lecture example (5% infection rate case)
counts = pd.DataFrame(
    {
        "Infected":      {"Positive": 490, "Negative": 10},
        "Not Infected":  {"Positive": 190, "Negative": 9310},
    }
)
counts

,Infected,Not Infected
Positive,490,190
Negative,10,9310


In [67]:
# 2) Compute Bayes exactly like in the lecture (from the table)
N = counts.values.sum()

pos_infected = counts.loc["Positive", "Infected"]
pos_total    = counts.loc["Positive"].sum()

P_I_given_Pos = pos_infected / pos_total

print("P(I|Pos) from table =", round(P_I_given_Pos, 4))
print("False positive rate among Pos =", round(1 - P_I_given_Pos, 4))

P(I|Pos) from table = 0.7206
False positive rate among Pos = 0.2794


In [68]:
# 3) Solve the task condition: infection rate falls to 1% (P(I)=0.01)
# Keep the same test characteristics as in the lecture:
# sensitivity P(Pos|I)=0.98, false positive P(Pos|NI)=0.02

P_I = 0.01
P_NI = 1 - P_I
P_Pos_given_I = 0.98
P_Pos_given_NI = 0.02

P_Pos = P_Pos_given_I * P_I + P_Pos_given_NI * P_NI
P_I_given_Pos_new = (P_Pos_given_I * P_I) / P_Pos

print("P(Pos) =", round(P_Pos, 4))
print("P(I|Pos) when P(I)=0.01 =", round(P_I_given_Pos_new, 4))
print("False positive rate among Pos =", round(1 - P_I_given_Pos_new, 4))

P(Pos) = 0.0296
P(I|Pos) when P(I)=0.01 = 0.3311
False positive rate among Pos = 0.6689




---

